# UberPulse AI
## AI-Powered Ride-Hailing Analytics & Demand Intelligence

**AICTE | IBM SkillsBuild Data Analytics with AI Internship Program 2026**

**Author:** Mohammed Fahad

This notebook presents the complete analytical workflow for UberPulse AI:
- Data loading and audit
- Data cleaning and feature engineering
- Exploratory data analysis
- Business KPI analysis
- Cancellation and operational analysis
- Leakage-aware pre-booking machine learning
- Model evaluation
- Business insights
- Conclusion

> **Dataset note:** This project uses a public NCR ride-bookings dataset. It is not proprietary or internal Uber data.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

BASE_DIR = Path.cwd()
RAW_PATH = BASE_DIR / "data" / "raw" / "ncr_ride_bookings.csv"
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "uberpulse_cleaned.csv"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)


## 2. Load Dataset

In [ ]:
# Use the raw dataset for the reproducible notebook workflow.
# If the notebook is opened from the notebooks/ folder, use the parent project directory.

if not RAW_PATH.exists():
    RAW_PATH = Path("../data/raw/ncr_ride_bookings.csv")

df = pd.read_csv(RAW_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Initial Data Audit

The audit checks structure, missing values, duplicates, booking outcomes, numeric ranges, and identifiers before cleaning.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percent", ascending=False)
display(missing)

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nBooking Status:")
display(df["Booking Status"].value_counts(dropna=False))

print("\nNumeric summary:")
display(df.describe(include="all").T)


### 3.1 Identifier and Validation Checks

In [ ]:
print("Unique Booking IDs:", df["Booking ID"].nunique())
print("Duplicate Booking IDs:", df["Booking ID"].duplicated().sum())
print("Unique Customer IDs:", df["Customer ID"].nunique())

numeric_candidates = [
    "Avg VTAT", "Avg CTAT", "Cancelled Rides by Customer",
    "Cancelled Rides by Driver", "Incomplete Rides",
    "Booking Value", "Ride Distance", "Driver Ratings",
    "Customer Rating"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nNegative numeric values:")
negative_counts = {}
for col in numeric_candidates:
    if col in df.columns:
        negative_counts[col] = int((df[col] < 0).sum())
display(pd.Series(negative_counts, name="Negative_Count"))

print("\nRating validation:")
for col in ["Driver Ratings", "Customer Rating"]:
    if col in df.columns:
        invalid = ((df[col].notna()) & ((df[col] < 1) | (df[col] > 5))).sum()
        print(f"{col}: {invalid} invalid values")


## 4. Data Cleaning and Feature Engineering

In [ ]:
clean = df.copy()

# Standardize string columns and remove surrounding quotes.
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype("string").str.strip().str.strip('"').str.strip("'")

# Numeric conversion.
for col in numeric_candidates:
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Parse date and time.
clean["DateTime"] = pd.to_datetime(
    clean["Date"].astype("string") + " " + clean["Time"].astype("string"),
    errors="coerce"
)

# Remove exact duplicate records only.
before = len(clean)
clean = clean.drop_duplicates().copy()
after = len(clean)

# Temporal features.
clean["Year"] = clean["DateTime"].dt.year
clean["Month"] = clean["DateTime"].dt.month
clean["Month_Name"] = clean["DateTime"].dt.month_name()
clean["Day"] = clean["DateTime"].dt.day
clean["Day_of_Week"] = clean["DateTime"].dt.dayofweek
clean["Day_Name"] = clean["DateTime"].dt.day_name()
clean["Hour"] = clean["DateTime"].dt.hour
clean["Is_Weekend"] = clean["Day_of_Week"].isin([5, 6])

def time_period(hour):
    if pd.isna(hour):
        return "Unknown"
    if 5 <= hour < 12:
        return "Morning"
    if 12 <= hour < 17:
        return "Afternoon"
    if 17 <= hour < 22:
        return "Evening"
    return "Night"

clean["Time_Period"] = clean["Hour"].apply(time_period)

# Business outcome flags.
clean["Is_Completed"] = (clean["Booking Status"] == "Completed").astype(int)
clean["Is_Cancelled"] = clean["Booking Status"].isin(
    ["Cancelled by Driver", "Cancelled by Customer"]
).astype(int)
clean["Is_Incomplete"] = (clean["Booking Status"] == "Incomplete").astype(int)
clean["Is_No_Driver_Found"] = (clean["Booking Status"] == "No Driver Found").astype(int)

clean["Booking_Outcome"] = clean["Booking Status"]

# Revenue is counted for completed rides only.
clean["Revenue"] = np.where(
    clean["Booking Status"].eq("Completed"),
    clean["Booking Value"].fillna(0),
    0
)

clean["Fare_Per_KM"] = np.where(
    clean["Ride Distance"].gt(0),
    clean["Booking Value"] / clean["Ride Distance"],
    np.nan
)

# Availability flags preserve the meaning of status-dependent missingness.
for col in [
    "Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance",
    "Driver Ratings", "Customer Rating", "Payment Method"
]:
    if col in clean.columns:
        clean[f"Has_{col.replace(' ', '_')}"] = clean[col].notna().astype(int)

print("Rows before cleaning:", before)
print("Rows after cleaning:", after)
print("Exact duplicate rows removed:", before - after)
print("Duplicate Booking IDs retained:", clean["Booking ID"].duplicated().sum())
print("Final columns:", clean.shape[1])

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED_PATH, index=False)

display(clean.head())


## 5. Booking Outcome Analysis

In [ ]:
status_counts = clean["Booking Status"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=status_counts.index, y=status_counts.values)
plt.title("Booking Status Distribution")
plt.xlabel("Booking Status")
plt.ylabel("Bookings")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

status_counts


## 6. Executive KPIs

In [ ]:
total_bookings = len(clean)
completed = int(clean["Is_Completed"].sum())
cancelled = int(clean["Is_Cancelled"].sum())
incomplete = int(clean["Is_Incomplete"].sum())
no_driver = int(clean["Is_No_Driver_Found"].sum())

kpis = {
    "Total Bookings": total_bookings,
    "Completed Bookings": completed,
    "Cancelled Bookings": cancelled,
    "Incomplete Bookings": incomplete,
    "No Driver Found": no_driver,
    "Completion Rate": completed / total_bookings,
    "Cancellation Rate": cancelled / total_bookings,
    "No Driver Found Rate": no_driver / total_bookings,
    "Incomplete Rate": incomplete / total_bookings,
    "Completed Ride Revenue": clean["Revenue"].sum(),
    "Average Booking Value": clean.loc[clean["Is_Completed"].eq(1), "Booking Value"].mean(),
    "Average Ride Distance": clean.loc[clean["Is_Completed"].eq(1), "Ride Distance"].mean(),
    "Average Driver Rating": clean["Driver Ratings"].mean(),
    "Average Customer Rating": clean["Customer Rating"].mean(),
}

for k, v in kpis.items():
    if "Rate" in k:
        print(f"{k}: {v:.2%}")
    elif isinstance(v, float):
        print(f"{k}: {v:,.2f}")
    else:
        print(f"{k}: {v:,}")


## 7. Demand and Time Analysis

In [ ]:
hourly = clean.groupby("Hour").size()

plt.figure(figsize=(12, 5))
plt.plot(hourly.index, hourly.values, marker="o")
plt.title("Bookings by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Bookings")
plt.xticks(range(24))
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

daily = clean.groupby("Day_Name").size()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = daily.reindex(day_order)

plt.figure(figsize=(10, 5))
sns.barplot(x=daily.index, y=daily.values)
plt.title("Bookings by Day of Week")
plt.xlabel("Day")
plt.ylabel("Bookings")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

period = clean["Time_Period"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=period.index, y=period.values)
plt.title("Bookings by Time Period")
plt.xlabel("Time Period")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()

print("Peak booking hour:", int(hourly.idxmax()))


## 8. Vehicle Performance

In [ ]:
vehicle = (
    clean.groupby("Vehicle Type")
    .agg(
        Bookings=("Booking ID", "size"),
        Completed=("Is_Completed", "sum"),
        Revenue=("Revenue", "sum")
    )
)
vehicle["Completion_Rate"] = vehicle["Completed"] / vehicle["Bookings"]
display(vehicle.sort_values("Bookings", ascending=False))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=vehicle.sort_values("Bookings", ascending=False).reset_index(),
    x="Vehicle Type",
    y="Bookings"
)
plt.title("Bookings by Vehicle Type")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 9. Location and Payment Analysis

In [ ]:
top_pickups = clean["Pickup Location"].value_counts().head(10)
top_drops = clean["Drop Location"].value_counts().head(10)
payment = clean["Payment Method"].value_counts()

print("Top pickup locations:")
display(top_pickups.to_frame("Bookings"))

print("Top drop locations:")
display(top_drops.to_frame("Bookings"))

print("Payment methods:")
display(payment.to_frame("Bookings"))

print("Top pickup location:", top_pickups.index[0])
print("Top payment method:", payment.index[0])


## 10. Cancellation Intelligence

In [ ]:
driver_data = clean[
    clean["Booking Status"].eq("Cancelled by Driver")
].copy()

customer_data = clean[
    clean["Booking Status"].eq("Cancelled by Customer")
].copy()

driver_reasons = (
    driver_data["Driver Cancellation Reason"]
    .astype("string").str.strip()
    .replace(["", "nan", "None"], pd.NA)
    .dropna()
    .value_counts()
    .head(10)
)

customer_reasons = (
    customer_data["Reason for cancelling by Customer"]
    .astype("string").str.strip()
    .replace(["", "nan", "None"], pd.NA)
    .dropna()
    .value_counts()
    .head(10)
)

print("Top driver cancellation reasons:")
display(driver_reasons.to_frame("Bookings"))

print("Top customer cancellation reasons:")
display(customer_reasons.to_frame("Bookings"))


## 11. Revenue Analysis

In [ ]:
completed_df = clean[clean["Booking Status"].eq("Completed")].copy()

revenue_by_vehicle = (
    completed_df.groupby("Vehicle Type")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    x=revenue_by_vehicle.index,
    y=revenue_by_vehicle.values
)
plt.title("Completed Ride Revenue by Vehicle Type")
plt.xlabel("Vehicle Type")
plt.ylabel("Revenue")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

monthly_revenue = (
    completed_df.groupby(["Year", "Month"])["Revenue"]
    .sum()
    .reset_index()
)

display(monthly_revenue)


## 12. Ratings and Ride Metrics

In [ ]:
rating_summary = clean[
    ["Driver Ratings", "Customer Rating", "Ride Distance", "Avg VTAT", "Avg CTAT"]
].describe().T

display(rating_summary)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=completed_df,
    x="Ride Distance",
    y="Booking Value",
    alpha=0.3
)
plt.title("Ride Distance vs Booking Value")
plt.xlabel("Ride Distance")
plt.ylabel("Booking Value")
plt.tight_layout()
plt.show()


## 13. Correlation Analysis

In [ ]:
corr_cols = [
    "Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance",
    "Driver Ratings", "Customer Rating", "Hour", "Is_Completed"
]

corr = clean[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 14. AI/ML: Leakage-Aware Pre-booking Prediction

A first wider model was intentionally not used as the main model because fields such as ride distance, ratings, and operational wait times may become available after a booking progresses.

The final model therefore uses request-stage or booking-context features:
- Vehicle Type
- Pickup Location
- Drop Location
- Hour
- Day_of_Week
- Is_Weekend
- Time_Period
- Payment Method

Target: **Completed vs Not Completed**.


In [ ]:
model_df = clean.copy()

target = "Is_Completed"

features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Hour",
    "Day_of_Week",
    "Is_Weekend",
    "Time_Period",
    "Payment Method"
]

model_data = model_df[features + [target]].copy()

# Remove rows missing the target only.
model_data = model_data.dropna(subset=[target])

X = model_data[features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Time_Period",
    "Payment Method"
]

numeric_features = [
    "Hour",
    "Day_of_Week",
    "Is_Weekend"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf)
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1": f1_score(y_test, pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, prob)
}

for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


## 15. Classification Report

In [ ]:
print(classification_report(
    y_test,
    pred,
    target_names=["Not Completed", "Completed"],
    digits=4,
    zero_division=0
))


## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Completed", "Completed"],
    yticklabels=["Not Completed", "Completed"]
)
plt.title("Pre-booking Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 17. Feature Importance

In [ ]:
# Retrieve transformed feature names and Random Forest importances.
pre = model.named_steps["preprocessor"]
clf = model.named_steps["classifier"]

feature_names = pre.get_feature_names_out()
importance = clf.feature_importances_

feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
)

display(feature_importance.head(20))

plt.figure(figsize=(10, 7))
top_fi = feature_importance.head(15).sort_values("Importance")
plt.barh(top_fi["Feature"], top_fi["Importance"])
plt.title("Top 15 Model Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 18. Save Model and ML Outputs

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
(REPORT_DIR / "ml").mkdir(exist_ok=True)

joblib.dump(model, MODEL_DIR / "prebooking_outcome_model.joblib")

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(REPORT_DIR / "ml" / "prebooking_metrics.csv", index=False)

predictions = X_test.copy()
predictions["Actual"] = y_test.values
predictions["Predicted"] = pred
predictions["Probability_Completed"] = prob
predictions.to_csv(
    REPORT_DIR / "ml" / "prebooking_predictions.csv",
    index=False
)

feature_importance.to_csv(
    REPORT_DIR / "ml" / "prebooking_feature_importance.csv",
    index=False
)

pd.DataFrame(
    cm,
    index=["Actual_Not_Completed", "Actual_Completed"],
    columns=["Predicted_Not_Completed", "Predicted_Completed"]
).to_csv(
    REPORT_DIR / "ml" / "prebooking_confusion_matrix.csv"
)

print("Model and ML reports saved.")


## 19. Business Insights Summary

In [ ]:
insights = {
    "total_bookings": int(total_bookings),
    "completed_bookings": int(completed),
    "cancellation_rate": round(cancelled / total_bookings, 4),
    "completion_rate": round(completed / total_bookings, 4),
    "no_driver_rate": round(no_driver / total_bookings, 4),
    "completed_revenue": float(clean["Revenue"].sum()),
    "peak_booking_hour": int(hourly.idxmax()),
    "top_vehicle": str(clean["Vehicle Type"].value_counts().idxmax()),
    "top_pickup_location": str(clean["Pickup Location"].value_counts().idxmax()),
    "top_payment_method": str(clean["Payment Method"].value_counts().idxmax()),
    "model_accuracy": round(metrics["Accuracy"], 4),
    "model_precision": round(metrics["Precision"], 4),
    "model_recall": round(metrics["Recall"], 4),
    "model_f1": round(metrics["F1"], 4),
    "model_roc_auc": round(metrics["ROC-AUC"], 4)
}

print(json.dumps(insights, indent=2))

(REPORT_DIR / "business_insights").mkdir(exist_ok=True)

with open(
    REPORT_DIR / "business_insights" / "business_insights.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(insights, f, indent=2)

print("Business insights saved.")


## 20. Conclusion

UberPulse AI provides an end-to-end Data Analytics with AI workflow from data auditing and cleaning to exploratory analysis, business KPIs, machine learning, and interactive dashboard delivery.

The final pre-booking Random Forest model is deliberately separated from the earlier wider model because post-outcome variables can create leakage. The resulting model provides a more defensible baseline for predicting booking completion using booking-context features.

The analysis can be extended with time-series demand forecasting, calibrated probabilities, route-level forecasting, model monitoring, and operational scenario analysis.


## 21. Project Outputs

The notebook produces or uses:
- `data/processed/uberpulse_cleaned.csv`
- `models/prebooking_outcome_model.joblib`
- `reports/ml/prebooking_metrics.csv`
- `reports/ml/prebooking_predictions.csv`
- `reports/ml/prebooking_feature_importance.csv`
- `reports/ml/prebooking_confusion_matrix.csv`
- `reports/business_insights/business_insights.json`

The Streamlit dashboard is provided separately in `app.py`.
